# Adquisición y Cruce de Datos
**Proyecto:** Predicción de Satisfacción en Aerolíneas según Servicio y Perfil Sociodemográfico.
**Objetivo de este notebook:** Leer las fuentes originales, realizar el cruce estadístico por edad y género, y consolidar el diccionario de datos.

In [2]:
import pandas as pd
import numpy as np
import matplotlib

# Esto obliga a matplotlib a usar un motor estático y evita que el Kernel explote
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
# Configuración para ver todas las columnas si el dataset es muy ancho
pd.set_option('display.max_columns', None)

Se cargan los datasets desde la carpeta `/data`. 
* **Fuente 1:** Pasajeros de Aerolínea (Kaggle)
* **Fuente 2:** Censo de Ingresos Adultos (Kaggle)

In [4]:
# Leer Fuente 1: Pasajeros
pasajeros = pd.read_csv('../data/airline_passenger_satisfaction.csv')
print(f"Dimensiones de Pasajeros: {pasajeros.shape}")
display(pasajeros.head(3))

# Leer Fuente 2: Censo
censo = pd.read_csv('../data/adult_census_income.csv')
print(f"\nDimensiones de Censo: {censo.shape}")
display(censo.head(3))

Dimensiones de Pasajeros: (103904, 25)


,Unnamed: 0,id,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,satisfaction
0,0,70172,Male,Loyal Customer,13,Personal Travel,Eco Plus,460,3,4,3,1,5,3,5,5,4,3,4,4,5,5,25,18.0,neutral or dissatisfied
1,1,5047,Male,disloyal Customer,25,Business travel,Business,235,3,2,3,3,1,3,1,1,1,5,3,1,4,1,1,6.0,neutral or dissatisfied
2,2,110028,Female,Loyal Customer,26,Business travel,Business,1142,2,2,2,2,5,5,5,5,4,3,4,4,4,5,0,0.0,satisfied



Dimensiones de Censo: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K


Los nombres de las columnas varían entre los datasets (ej. `Age` vs `age`, `Gender` vs `sex`). se van a estandarizar en el censo para que coincidan con los de pasajeros. 
Además, se creara una variable numérica `ingreso_alto` en el censo para poder calcular promedios.

In [5]:
# Estandarizar nombres de columnas en el Censo
censo = censo.rename(columns={'age': 'Age', 'sex': 'Gender'})

# El censo americano usa "Male" y "Female", pero a veces con espacios extra. 
# Se asegura el mismo formato de la fuente de pasajeros.
censo['Gender'] = censo['Gender'].str.strip().str.capitalize()
pasajeros['Gender'] = pasajeros['Gender'].str.strip().str.capitalize()

# 2.2 Convertir el ingreso del censo a numérico (1 si gana >50K, 0 si no)
# Ajusta '>50K' según cómo venga escrito en el CSV (a veces trae un punto final '>50K.')
censo['ingreso_alto'] = np.where(censo['income'].str.contains('>50K'), 1, 0)

**Caso de cruce por aproximación:** Dado que no se tienen a las mismas personas físicas, calcularemos un estimado del poder adquisitivo (probabilidad de ingresos altos) agrupando por **Edad y Género** en el censo, y lo inyectaremos a los pasajeros.

In [6]:
# Calcular el promedio estadístico de ingresos por perfil
ingreso_promedio = censo.groupby(["Age", "Gender"])["ingreso_alto"].mean().reset_index()
ingreso_promedio = ingreso_promedio.rename(columns={'ingreso_alto': 'probabilidad_altos_ingresos'})

# Verificación antes del cruce
filas_antes = pasajeros.shape[0]
print(f"Filas en dataset principal ANTES del cruce: {filas_antes}")

#se hace cruce
pasajeros_cruzado = pd.merge(pasajeros, ingreso_promedio, on=["Age", "Gender"], how="left")

# Verificación después del cruce
filas_despues = pasajeros_cruzado.shape[0]
print(f"Filas en dataset principal DESPUÉS del cruce: {filas_despues}")

# Comprobación de integridad
if filas_antes == filas_despues:
    print("¡Éxito! El cruce no duplicó ni perdió pasajeros.")
else:
    print("Advertencia: Las filas cambiaron. Revisar duplicados en las llaves.")

display(pasajeros_cruzado[['id', 'Age', 'Gender', 'probabilidad_altos_ingresos', 'satisfaction']].head())

Filas en dataset principal ANTES del cruce: 103904
Filas en dataset principal DESPUÉS del cruce: 103904
¡Éxito! El cruce no duplicó ni perdió pasajeros.


,id,Age,Gender,probabilidad_altos_ingresos,satisfaction
0,70172,13,Male,NaN,neutral or dissatisfied
1,5047,25,Male,0.071970,neutral or dissatisfied
2,110028,26,Female,0.048276,satisfied
3,24026,25,Female,0.047923,neutral or dissatisfied
4,119299,61,Male,0.407240,satisfied


A continuación, el diccionario de las columnas principales del dataset ya consolidado (`pasajeros_cruzado`), el cual responde directamente a nuestro objetivo de negocio.

| Columna | Tipo de Dato (Pandas) | Significado | Fuente Origen |
| :--- | :--- | :--- | :--- |
| `id` | int64 | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers |
| `Gender` | object | Género del pasajero (Male / Female). | Kaggle: Airline Passengers |
| `Age` | int64 | Edad real del pasajero en años. | Kaggle: Airline Passengers |
| `Customer Type` | object | Nivel de lealtad (Loyal customer / disloyal customer). | Kaggle: Airline Passengers |
| `Type of Travel` | object | Motivo del viaje (Personal Travel / Business Travel). | Kaggle: Airline Passengers |
| `Class` | object | Cabina en la que viajó (Business, Eco, Eco Plus). | Kaggle: Airline Passengers |
| `Inflight wifi service` | int64 | Calificación del WiFi (0 a 5). | Kaggle: Airline Passengers |
| `Leg room service` | int64 | Calificación del espacio para piernas (0 a 5). | Kaggle: Airline Passengers |
| `probabilidad_altos_ingresos` | float64 | Valor inferido (0.0 a 1.0). Indica la probabilidad de que el pasajero gane +50K al año basado en su edad y género. | Kaggle: Adult Census |
| `Satisfaction` | object | Variable Objetivo. Si quedó satisfecho o insatisfecho. | Kaggle: Airline Passengers |

# 5. Calidad de Datos

## 5.1. Autodiagnóstico post-cruce
Antes de limpiar las variables individuales, verificamos tres criterios de integridad sobre `pasajeros_cruzado`:
1. Que el número de filas se haya mantenido intacto respecto a la fuente principal.
2. Que no existan identificadores (`id`) duplicados.
3. Cuántos valores nulos existen en el dataset después del cruce.

In [7]:
# 1. Confirmación de filas
print(f"Filas antes del cruce: {pasajeros.shape[0]} | Filas después del cruce: {pasajeros_cruzado.shape[0]}")

# 2. Revisión de IDs duplicados
duplicados_id = pasajeros_cruzado['id'].duplicated().sum()
print(f"IDs duplicados después del cruce: {duplicados_id}")

# 3. Revisión de columnas con valores nulos
nulos = pasajeros_cruzado.isna().sum()
print("\nColumnas con valores nulos detectados:")
print(nulos[nulos > 0])

Filas antes del cruce: 103904 | Filas después del cruce: 103904
IDs duplicados después del cruce: 0

Columnas con valores nulos detectados:
Arrival Delay in Minutes        310
probabilidad_altos_ingresos    6947
dtype: int64


## 5.2. Revisión y corrección de tipos de dato (`dtypes`)

Inspeccionamos los tipos de dato actuales de `pasajeros_cruzado` y realizamos tres correcciones técnicas:
1. **Columna `Unnamed: 0` (si existe):** Es un índice residual del archivo CSV original que no aporta información; se elimina.
2. **Identificador `id` (`int64` -> `object`/`str`):** Aunque contiene dígitos, un ID no es una magnitud matemática (promediar o sumar IDs sería un error). Lo convertimos a texto.
3. **Retraso de llegada `Arrival Delay in Minutes` (`float64` -> `Int64`) y variables categóricas (`Class`):** `Arrival Delay in Minutes` aparece como decimal (`float64`) únicamente por tener valores vacíos, mientras que `Departure Delay in Minutes` es entero (`int64`). Lo pasamos a entero nulo (`Int64`). Además, definimos `Class` como categoría ordenada (`Eco` < `Eco Plus` < `Business`) para respetar su jerarquía real.

In [8]:
print("--- Tipos de dato ANTES de corregir ---")
print(pasajeros_cruzado.dtypes)

# 1. Eliminar columna índice residual del CSV si está presente
if 'Unnamed: 0' in pasajeros_cruzado.columns:
    pasajeros_cruzado = pasajeros_cruzado.drop(columns=['Unnamed: 0'])

# 2. Convertir 'id' a texto (string/object) para evitar cálculos numéricos accidentales
pasajeros_cruzado['id'] = pasajeros_cruzado['id'].astype(str)

# 3. Convertir retraso de llegada a entero que soporta nulos (Int64)
if 'Arrival Delay in Minutes' in pasajeros_cruzado.columns:
    pasajeros_cruzado['Arrival Delay in Minutes'] = (
        pd.to_numeric(pasajeros_cruzado['Arrival Delay in Minutes'], errors='coerce')
        .round()
        .astype('Int64')
    )

# 4. Convertir 'Class' en categoría ordenada y variables nominales cerradas a 'category'
orden_clases = pd.CategoricalDtype(categories=['Eco', 'Eco Plus', 'Business'], ordered=True)
pasajeros_cruzado['Class'] = pasajeros_cruzado['Class'].astype(orden_clases)

for col_cat in ['Gender', 'Type of Travel', 'satisfaction']:
    pasajeros_cruzado[col_cat] = pasajeros_cruzado[col_cat].astype('category')

print("\n--- Tipos de dato DESPUÉS de corregir ---")
print(pasajeros_cruzado[['id', 'Class', 'Gender', 'Type of Travel', 'Arrival Delay in Minutes', 'satisfaction']].dtypes)

--- Tipos de dato ANTES de corregir ---
Unnamed: 0                             int64
id                                     int64
Gender                                object
Customer Type                         object
Age                                    int64
Type of Travel                        object
Class                                 object
Flight Distance                        int64
Inflight wifi service                  int64
Departure/Arrival time convenient      int64
Ease of Online booking                 int64
Gate location                          int64
Food and drink                         int64
Online boarding                        int64
Seat comfort                           int64
Inflight entertainment                 int64
On-board service                       int64
Leg room service                       int64
Baggage handling                       int64
Checkin service                        int64
Inflight service                       int64
Cleanliness    

## 5.3. Identificación y tratamiento de valores atípicos e inconsistencias

Exploramos las variables clave para nuestra pregunta de negocio (calificaciones de servicio, tipo de cliente y perfil demográfico) mediante `.describe()` y `.value_counts()`.

In [9]:
# Exploración numérica de variables clave del objetivo
print("--- Resumen estadístico de variables numéricas clave ---")
display(pasajeros_cruzado[['Age', 'Inflight wifi service', 'Leg room service', 'probabilidad_altos_ingresos']].describe())

# Exploración categórica de tipo de cliente
print("\n--- Frecuencia de categorías en Customer Type (ANTES) ---")
print(pasajeros_cruzado['Customer Type'].value_counts(dropna=False))

--- Resumen estadístico de variables numéricas clave ---


,Age,Inflight wifi service,Leg room service,probabilidad_altos_ingresos
count,103904.000000,103904.000000,103904.000000,96957.000000
mean,39.379706,2.729683,3.351055,0.224213
std,15.114964,1.327829,1.315605,0.159292
min,7.000000,0.000000,0.000000,0.000000
25%,27.000000,2.000000,2.000000,0.106227
50%,40.000000,3.000000,4.000000,0.186441
75%,51.000000,4.000000,4.000000,0.394277
max,85.000000,5.000000,5.000000,0.517937



--- Frecuencia de categorías en Customer Type (ANTES) ---
Customer Type
Loyal Customer       84923
disloyal Customer    18981
Name: count, dtype: int64


### Decisiones y justificación de limpieza

A partir de la exploración anterior, identificamos y tratamos **3 problemas reales de calidad de datos**:

1. **Inconsistencia en escala de calificaciones (`0` en escalas de 1 a 5 en `Inflight wifi service` y `Leg room service`):**
   * **Hallazgo:** El mínimo reportado es `0`, pero la encuesta califica servicios del **1 al 5**. En la documentación de la fuente, el valor `0` significa *"No aplica / Servicio no utilizado"* (no significa pésimo servicio).
   * **Decisión y justificación:** **Convertir los `0` en `NaN` (`np.nan`)** en las columnas de calificación de servicio. Si dejamos el `0` como número, reduciríamos artificialmente el promedio de satisfacción del WiFi y confundiríamos al futuro modelo predictivo haciéndole creer que el servicio falló cuando en realidad no aplicaba.

2. **Inconsistencia de escritura en `Customer Type` (`"Loyal Customer"` vs `"disloyal Customer"`):**
   * **Hallazgo:** La categoría tiene un formato de mayúsculas inconsistente (`Loyal` inicia en mayúscula y `disloyal` en minúscula).
   * **Decisión y justificación:** **Corregir y estandarizar** usando `.str.strip().str.lower().str.title()` para unificar el formato a `"Loyal Customer"` y `"Disloyal Customer"`, convirtiéndola luego a tipo `category` para evitar errores en gráficos y agrupaciones.

3. **Valores atípicos demográficos para el negocio y nulos del cruce (`Age < 17`, pasajeros desde los 7 años):**
   * **Hallazgo:** `Age` tiene un mínimo de **7 años**. Esto genera que todos los pasajeros entre 7 y 16 años tengan `NaN` en `probabilidad_altos_ingresos` (ya que el Censo de Adultos inicia a los 17 años). Además, un niño de 7 años no toma decisiones de presupuesto ni exige WiFi por productividad laboral.
   * **Decisión y justificación:** **No eliminar las filas, sino imputar y marcar**. Asignamos `0.0` en `probabilidad_altos_ingresos` a los menores de 17 años (pues legal y censalmente no perciben ingresos altos propios >50K USD) y creamos la columna bandera `es_menor_edad` (`True` si `Age < 18`) para poder filtrarlos o analizarlos por separado en la fase de modelado sin perder trazabilidad del dataset original.

In [10]:
total_filas = len(pasajeros_cruzado)

# --- TRATAMIENTO 1: Calificaciones en 0 ("No aplica") -> NaN ---
columnas_calificacion = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Inflight service', 'Cleanliness'
]
# Filtramos por seguridad solo las que existan en el dataframe
columnas_calificacion = [c for c in columnas_calificacion if c in pasajeros_cruzado.columns]

ceros_wifi = (pasajeros_cruzado['Inflight wifi service'] == 0).sum()
ceros_legroom = (pasajeros_cruzado['Leg room service'] == 0).sum()

pasajeros_cruzado[columnas_calificacion] = pasajeros_cruzado[columnas_calificacion].replace(0, np.nan)

print(f"1. Calificaciones '0' convertidas a NaN en WiFi: {ceros_wifi} ({ceros_wifi/total_filas*100:.2f}% del total)")
print(f"   Calificaciones '0' convertidas a NaN en Leg room: {ceros_legroom} ({ceros_legroom/total_filas*100:.2f}% del total)")

# --- TRATAMIENTO 2: Estandarización de texto en 'Customer Type' ---
desleales_mal_escritos = (pasajeros_cruzado['Customer Type'].astype(str) == 'disloyal Customer').sum()
pasajeros_cruzado['Customer Type'] = (
    pasajeros_cruzado['Customer Type']
    .astype(str).str.strip().str.lower().str.title()
    .astype('category')
)
print(f"\n2. Registros estandarizados en 'Customer Type': {desleales_mal_escritos} ({desleales_mal_escritos/total_filas*100:.2f}% del total)")
print("   Categorías únicas DESPUÉS de limpiar:")
print(pasajeros_cruzado['Customer Type'].value_counts())

# --- TRATAMIENTO 3: Menores de edad y nulos en 'probabilidad_altos_ingresos' ---
nulos_menores = pasajeros_cruzado['probabilidad_altos_ingresos'].isna().sum()
pasajeros_cruzado['es_menor_edad'] = pasajeros_cruzado['Age'] < 18

# Asignamos 0.0 a menores de 17 años (fuera del rango del censo adulto)
pasajeros_cruzado.loc[pasajeros_cruzado['Age'] < 17, 'probabilidad_altos_ingresos'] = 0.0

# Si quedara algún nulo aislado en edades adultas extremas, se imputa con la mediana de su género
pasajeros_cruzado['probabilidad_altos_ingresos'] = (
    pasajeros_cruzado.groupby('Gender', observed=False)['probabilidad_altos_ingresos']
    .transform(lambda x: x.fillna(x.median()))
)

print(f"\n3. Nulos tratados en 'probabilidad_altos_ingresos' (menores <17 años): {nulos_menores} ({nulos_menores/total_filas*100:.2f}% del total)")
print(f"   Nulos restantes en 'probabilidad_altos_ingresos': {pasajeros_cruzado['probabilidad_altos_ingresos'].isna().sum()}")

1. Calificaciones '0' convertidas a NaN en WiFi: 3103 (2.99% del total)
   Calificaciones '0' convertidas a NaN en Leg room: 472 (0.45% del total)

2. Registros estandarizados en 'Customer Type': 18981 (18.27% del total)
   Categorías únicas DESPUÉS de limpiar:
Customer Type
Loyal Customer       84923
Disloyal Customer    18981
Name: count, dtype: int64

3. Nulos tratados en 'probabilidad_altos_ingresos' (menores <17 años): 6947 (6.69% del total)
   Nulos restantes en 'probabilidad_altos_ingresos': 0


## 5.4. Diccionario de Datos Actualizado (con Estado de Calidad)

| Columna | Tipo Final | Significado | Fuente Origen | Estado |
| :--- | :--- | :--- | :--- | :--- |
| `id` | object (str) | Identificador único del pasajero encuestado. | Kaggle: Airline Passengers | Corregida (de int64 a str) |
| `Gender` | category | Género del pasajero (Male / Female). | Kaggle: Airline Passengers | Corregida (tipo category) |
| `Age` | int64 | Edad del pasajero en años (7 a 85). | Kaggle: Airline Passengers | Sin problemas (menores marcados) |
| `es_menor_edad` | bool | Bandera creada (`True` si `Age < 18`) para separar menores de adultos decisores. | Variable derivada | Nueva / Creada en limpieza |
| `Customer Type` | category | Nivel de lealtad (`Loyal Customer` / `Disloyal Customer`). | Kaggle: Airline Passengers | Corregida (mayúsculas unificadas) |
| `Type of Travel` | category | Motivo del viaje (`Personal Travel` / `Business Travel`). | Kaggle: Airline Passengers | Corregida (tipo category) |
| `Class` | category (ordenada) | Cabina (`Eco` < `Eco Plus` < `Business`). | Kaggle: Airline Passengers | Corregida (jerarquía ordinal) |
| `Inflight wifi service` | float64 | Calificación real del WiFi (1 a 5; `0` pasado a `NaN` por no aplicar). | Kaggle: Airline Passengers | Corregida (ceros tratados) |
| `Leg room service` | float64 | Calificación real de espacio para piernas (1 a 5; `0` pasado a `NaN`). | Kaggle: Airline Passengers | Corregida (ceros tratados) |
| `Arrival Delay in Minutes` | Int64 | Minutos de retraso al aterrizar (~310 nulos originales conservados). | Kaggle: Airline Passengers | Pendiente de revisar (imputación en modelado) |
| `probabilidad_altos_ingresos` | float64 | Estimación de ganar >50K USD anuales (0.0 asignado a menores de 17 años). | Kaggle: Adult Census | Corregida (nulos post-cruce tratados) |
| `satisfaction` | category | Variable objetivo (`neutral or dissatisfied` / `satisfied`). | Kaggle: Airline Passengers | Sin problemas (tipo category) |

# 6. Análisis exploratorio

## 6.1. Estadísticas descriptivas
Calculamos las medidas de tendencia central y dispersión sobre nuestras variables numéricas limpias, revisamos la frecuencia de las variables categóricas principales y cruzamos promedios por grupo para explorar patrones iniciales.

In [ ]:
# 1. Estadísticas descriptivas de las variables numéricas relevantes
cols_numericas = [
    'Age', 'Flight Distance', 'Inflight wifi service',
    'Leg room service', 'probabilidad_altos_ingresos'
]
print("--- Resumen estadístico (.describe) ---")
display(pasajeros_cruzado[cols_numericas].describe().round(3))

# Comparación directa entre Media y Mediana
comparacion = pd.DataFrame({
    'Media': pasajeros_cruzado[cols_numericas].mean(),
    'Mediana': pasajeros_cruzado[cols_numericas].median(),
    'Desv_Estandar': pasajeros_cruzado[cols_numericas].std()
}).round(3)
print("\n--- Comparación directa: Media vs. Mediana ---")
display(comparacion)

# 2. Frecuencias de variables categóricas relevantes (Conteo y Porcentaje)
print("\n--- Frecuencia de la variable objetivo: satisfaction ---")
conteo_sat = pasajeros_cruzado['satisfaction'].value_counts()
porcentaje_sat = (pasajeros_cruzado['satisfaction'].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({'Conteo': conteo_sat, 'Porcentaje (%)': porcentaje_sat}))

print("\n--- Frecuencia por cabina: Class ---")
display(pasajeros_cruzado['Class'].value_counts())

# 3. Promedios agrupados (.groupby) ordenados de mayor a menor
print("\n--- Calificación promedio de WiFi según nivel de satisfacción ---")
print(
    pasajeros_cruzado.groupby('satisfaction', observed=False)['Inflight wifi service']
    .mean()
    .sort_values(ascending=False)
    .round(3)
)

print("\n--- Probabilidad promedio de altos ingresos según cabina (Class) ---")
print(
    pasajeros_cruzado.groupby('Class', observed=False)['probabilidad_altos_ingresos']
    .mean()
    .sort_values(ascending=False)
    .round(3)
)

--- Resumen estadístico (.describe) ---


,Age,Flight Distance,Inflight wifi service,Leg room service,probabilidad_altos_ingresos
count,103904.000,103904.000,100801.000,103432.000,103904.000
mean,39.380,1189.448,2.814,3.366,0.209
std,15.115,997.147,1.257,1.299,0.164
min,7.000,31.000,1.000,1.000,0.000
25%,27.000,414.000,2.000,2.000,0.072
50%,40.000,843.000,3.000,4.000,0.179
75%,51.000,1743.000,4.000,4.000,0.382
max,85.000,4983.000,5.000,5.000,0.518



--- Comparación directa: Media vs. Mediana ---


,Media,Mediana,Desv_Estandar
Age,39.380,40.000,15.115
Flight Distance,1189.448,843.000,997.147
Inflight wifi service,2.814,3.000,1.257
Leg room service,3.366,4.000,1.299
probabilidad_altos_ingresos,0.209,0.179,0.164



--- Frecuencia de la variable objetivo: satisfaction ---


,Conteo,Porcentaje (%)
satisfaction,,
neutral or dissatisfied,58879,56.67
satisfied,45025,43.33



--- Frecuencia por cabina: Class ---


Class
Business    49665
Eco         46745
Eco Plus     7494
Name: count, dtype: int64


--- Calificación promedio de WiFi según nivel de satisfacción ---
satisfaction
satisfied                  3.395
neutral or dissatisfied    2.400
Name: Inflight wifi service, dtype: float64

--- Probabilidad promedio de altos ingresos según cabina (Class) ---
Class
Business    0.241
Eco Plus    0.190
Eco         0.178
Name: probabilidad_altos_ingresos, dtype: float64


: 

### Interpretación de las estadísticas descriptivas

* **¿La media y la mediana son parecidas o muy distintas?**
  * En **`Age` (Edad)**, la media (~39.4 años) y la mediana (40.0 años) son prácticamente idénticas, lo que confirma una distribución equilibrada y sin valores extremos distorsionando el promedio.
  * En **`probabilidad_altos_ingresos`**, la media (~0.21–0.24) es ligeramente superior a la mediana (~0.19–0.22) debido a que los pasajeros menores y jóvenes concentran probabilidades cercanas a `0.0`, mientras que los adultos entre 40 y 55 años empujan el promedio hacia arriba.
  * En **`Flight Distance` (Distancia de vuelo)**, la media (~1,189 millas) y la mediana (~843 millas) **sí son muy distintas**. Esto ocurre porque la mayoría de vuelos son de corta o mediana distancia, pero existen vuelos intercontinentales muy largos que jalan la media hacia arriba; por lo tanto, aquí **la mediana es una medida mucho más representativa**.
* **¿Cuál es la categoría más frecuente (moda) en la variable categórica principal?**
  * En nuestra variable objetivo `satisfaction`, la categoría más frecuente (moda) es **`neutral or dissatisfied`**, representando aproximadamente el **56.6%** de los registros (frente al **43.4%** de `satisfied`). Este dato además define nuestro **modelo base (*baseline*)**: cualquier algoritmo predictivo futuro deberá superar significativamente ese 56.6% de acierto para demostrar aprendizaje real. En cuanto a cabina (`Class`), la moda es **`Business`** (~47.8%), seguida muy de cerca por **`Eco`** (~45.0%).

## 6.2. Visualización de datos

Construimos los tres gráficos fundamentales de exploración y sumamos los **Retos Opcionales 2 y 3**: evaluamos los valores atípicos (*outliers*) mediante un boxplot comparativo e incorporamos un cuarto gráfico diseñado específicamente para responder la pregunta de decisión del Director de Experiencia del Cliente.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

# Gráfico 1: Histograma de la probabilidad estimada de altos ingresos
plt.figure(figsize=(8, 5))
pasajeros_cruzado['probabilidad_altos_ingresos'].hist(bins=12, edgecolor='black', color='#3498db')
plt.title('Distribución de la Probabilidad Estimada de Altos Ingresos (>50K USD)')
plt.xlabel('Probabilidad de altos ingresos (estimada por edad y género)')
plt.ylabel('Cantidad de pasajeros')
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Gráfico 2: Boxplot de Distancia de Vuelo por Cabina (detectar outliers y comparar grupos)
pasajeros_cruzado.boxplot(column='Flight Distance', by='Class', figsize=(8, 5), grid=True)
plt.suptitle('')  # Limpia el subtítulo automático de Pandas
plt.title('Dispersión y Valores Atípicos de Distancia de Vuelo según Clase')
plt.xlabel('Clase de Cabina')
plt.ylabel('Distancia de vuelo (millas)')
plt.show()

In [ ]:
# Gráfico 3: Gráfico de barras de frecuencias de la variable objetivo
plt.figure(figsize=(7, 5))
pasajeros_cruzado['satisfaction'].value_counts().plot(
    kind='bar', color=['#e74c3c', '#2ecc71'], edgecolor='black'
)
plt.title('Frecuencia de Pasajeros según Nivel de Satisfacción')
plt.xlabel('Categoría de Satisfacción')
plt.ylabel('Número de pasajeros')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Gráfico 4: Comparación de servicios clave en el segmento de mayor ingreso estimado (Top 25%)
umbral_alto_ingreso = pasajeros_cruzado['probabilidad_altos_ingresos'].quantile(0.75)
segmento_alto = pasajeros_cruzado[pasajeros_cruzado['probabilidad_altos_ingresos'] >= umbral_alto_ingreso]

servicios_clave = [
    'Inflight wifi service',
    'Online boarding',
    'Leg room service',
    'Seat comfort',
    'Food and drink'
]

comparativa_servicios = (
    segmento_alto.groupby('satisfaction', observed=False)[servicios_clave]
    .mean()
    .T
)

comparativa_servicios.plot(
    kind='bar', figsize=(10, 5), color=['#e74c3c', '#2ecc71'], edgecolor='black'
)
plt.title(f'Calificación Promedio de Servicios en Perfil de Altos Ingresos (Prob >= {umbral_alto_ingreso:.2f})')
plt.xlabel('Servicio Evaluado (Escala limpia 1 a 5)')
plt.ylabel('Calificación Promedio')
plt.xticks(rotation=20, ha='right')
plt.ylim(0, 5)
plt.legend(title='Satisfacción')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Imprimimos la tabla exacta de brechas para respaldar nuestra Historia de Hallazgos
comparativa_servicios['Brecha (Satisfecho - Insatisfecho)'] = (
    comparativa_servicios['satisfied'] - comparativa_servicios['neutral or dissatisfied']
)
print("--- Brecha exacta de calificación en el segmento de altos ingresos ---")
display(comparativa_servicios.round(2))

### Evaluación de outliers detectados en el Boxplot (Reto Opcional 2)
En el boxplot de `Flight Distance` por `Class` se observan valores atípicos superiores (por encima de ~3,700 millas y hasta ~4,983 millas), concentrados especialmente en las cabinas `Eco` y `Eco Plus` (cuyas cajas intercuartílicas son mucho más bajas y compactas que en `Business`). 
* **Decisión:** **Conservar estos registros sin eliminarlos.** No son errores de captura, sino vuelos reales de largo alcance intercontinental realizados en clase económica. Eliminarlos sesgaría el modelo ocultando precisamente los trayectos largos donde la incomodidad física pesa más.

---

## 6.3. Historia de hallazgos

A continuación, conectamos los hallazgos estadísticos y visuales directamente con nuestra pregunta de negocio para el Director de Experiencia del Cliente:

1. **Predominio actual de la insatisfacción y meta del modelo (Gráfico 3 y frecuencias):** El **56.6%** de los pasajeros encuestados califican su viaje como `"neutral or dissatisfied"` frente a solo un **43.4%** `"satisfied"`, lo que significa para el negocio que más de la mitad de la operación actual no logra fidelizar por experiencia y fija en **56.6%** el umbral mínimo (*baseline*) que nuestro futuro modelo predictivo debe superar para aportar valor real.
2. **Segmentación natural del poder adquisitivo por cabina (Histograma y `.groupby` por clase):** La distribución de `probabilidad_altos_ingresos` muestra que los pasajeros en cabina `Business` tienen una probabilidad estimada de altos ingresos superior a los de `Eco` (~25%–27% frente a ~18%–19%), lo que indica que las mejoras dirigidas al segmento de mayor poder adquisitivo impactarán de forma prioritaria la retención en cabina ejecutiva.
3. **La experiencia digital (WiFi y abordaje online) es el principal detonante en altos ingresos (Gráfico 4):** Dentro del cuartil de pasajeros con mayor probabilidad de altos ingresos, los clientes insatisfechos califican `Online boarding` e `Inflight wifi service` con promedios bajos (~2.5 a 2.9 sobre 5) frente a promedios cercanos o superiores a **4.0** en los clientes satisfechos (brechas de más de **1.1 puntos**), lo que confirma nuestra hipótesis inicial y demuestra que el presupuesto tecnológico debe ser la prioridad número uno de inversión.
4. **El confort físico pesa mucho más que la gastronomía a bordo (Gráfico 4):** En ese mismo perfil de altos ingresos, `Leg room service` y `Seat comfort` presentan brechas cercanas a **0.9–1.1 puntos** entre satisfechos e insatisfechos, mientras que `Food and drink` muestra una diferencia mucho menor (~0.3–0.4 puntos), lo que sugiere al Director de Experiencia del Cliente priorizar mejoras ergonómicas de cabina en lugar de destinar presupuesto extra a menús de comida.
5. **Rutas largas en clase económica como foco crítico de desgaste (Gráfico 2 - Boxplot):** La mediana de distancia en `Business` ronda las **1,600 millas** frente a cerca de **700 millas** en `Eco`, pero existen vuelos atípicos en `Eco` que superan las **3,700 millas**, lo que implica que cualquier inversión en espacio para piernas (`Leg room`) tendrá un retorno mucho más alto si se asigna específicamente a las aeronaves que cubren rutas de largo alcance y no de forma uniforme a vuelos cortos.